In [ ]:
# works well for with ppe
import cv2
import numpy as np
import time
from ultralytics import YOLO
from collections import defaultdict, deque

# =========================
# CONFIG
# =========================
MODEL_PATH = "best.pt"
VIDEO_PATH = "rd1.mp4"

CONF_THRES = 0.25
IOU_THRES_VEST = 0.15

PPE_BUFFER = 20
PPE_CONFIRM_FRAMES = 4
MISSING_PERSIST_FRAMES = 15
ALERT_COOLDOWN_SEC = 10

# =========================
# DANGER ZONE (POLYGON)
# =========================
DANGER_ZONE = np.array(
    [(276, 229), (426, 229), (428, 210), (571, 205),
     (640, 430), (420, 447), (417, 474), (184, 468)],
    dtype=np.int32
)

# =========================
# LOAD MODEL
# =========================
model = YOLO(MODEL_PATH)

# =========================
# VIDEO
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)

# =========================
# MEMORY
# =========================
ppe_memory = defaultdict(lambda: {
    "helmet": deque(maxlen=PPE_BUFFER),
    "vest": deque(maxlen=PPE_BUFFER)
})

missing_counter = defaultdict(int)
last_alert_time = {}

# =========================
# HELPER FUNCTIONS
# =========================
def point_inside_polygon(point, polygon):
    return cv2.pointPolygonTest(polygon, point, False) >= 0

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

def box_center(box):
    return ((box[0] + box[2]) // 2, (box[1] + box[3]) // 2)

def point_inside_box(point, box):
    x, y = point
    return box[0] <= x <= box[2] and box[1] <= y <= box[3]

def get_head_box(person):
    x1, y1, x2, y2 = person
    h = y2 - y1
    w = x2 - x1
    return (
        int(x1 - 0.10 * w),
        int(y1),
        int(x2 + 0.10 * w),
        int(y1 + 0.40 * h)
    )

def get_torso_box(person):
    x1, y1, x2, y2 = person
    h = y2 - y1
    w = x2 - x1
    return (
        int(x1 - 0.15 * w),
        int(y1 + 0.25 * h),
        int(x2 + 0.15 * w),
        int(y1 + 0.80 * h)
    )

# =========================
# MAIN LOOP
# =========================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    now = time.time()

    results = model.track(frame, persist=True, conf=CONF_THRES)

    persons, helmets, vests = [], [], []

    for r in results:
        if r.boxes.id is None:
            continue

        for box, cls, tid in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.id):
            label = model.names[int(cls)]
            box = box.cpu().numpy().astype(int)
            tid = int(tid)

            if label == "person":
                persons.append((box, tid))
            elif label == "helmet":
                helmets.append(box)
            elif label == "safety-vest":
                vests.append(box)

    # Draw danger zone
    cv2.polylines(frame, [DANGER_ZONE], True, (0, 0, 255), 3)

    for person_box, pid in persons:
        x1, y1, x2, y2 = person_box
        foot_point = (int((x1 + x2) / 2), int(y2))

        inside_zone = point_inside_polygon(foot_point, DANGER_ZONE)

        head_box = get_head_box(person_box)
        torso_box = get_torso_box(person_box)

        # Helmet: center-point logic (robust to tilt)
        has_helmet = any(
            point_inside_box(box_center(h), head_box)
            for h in helmets
        )

        # Vest: IoU OR center-point
        has_vest = any(
            iou(torso_box, v) > IOU_THRES_VEST or
            point_inside_box(box_center(v), torso_box)
            for v in vests
        )

        # Temporal PPE memory
        ppe_memory[pid]["helmet"].append(has_helmet)
        ppe_memory[pid]["vest"].append(has_vest)

        helmet_score = sum(ppe_memory[pid]["helmet"])
        vest_score = sum(ppe_memory[pid]["vest"])

        helmet_ok = helmet_score >= PPE_CONFIRM_FRAMES
        vest_ok = vest_score >= PPE_CONFIRM_FRAMES

        missing = []
        if not helmet_ok:
            missing.append("Helmet")
        if not vest_ok:
            missing.append("Vest")

        if inside_zone and missing:
            missing_counter[pid] += 1
        else:
            missing_counter[pid] = 0

        violation = missing_counter[pid] > MISSING_PERSIST_FRAMES

        # Visualization
        if violation:
            color = (0, 0, 255)
            label = f"ID {pid}: MISSING {' & '.join(missing)}"

            last_time = last_alert_time.get(pid, 0)
            if now - last_time > ALERT_COOLDOWN_SEC:
                print(f"🚨 ALERT | ID {pid} | Missing: {missing}")
                last_alert_time[pid] = now
        else:
            color = (0, 255, 0)
            label = f"ID {pid}: SAFE"

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.circle(frame, foot_point, 5, (255, 0, 0), -1)
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # Debug (optional – comment if not needed)
        cv2.rectangle(frame, head_box[:2], head_box[2:], (255, 255, 0), 1)
        cv2.rectangle(frame, torso_box[:2], torso_box[2:], (0, 255, 255), 1)

    cv2.imshow("PPE Danger Zone Monitoring", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 5 persons, 1 helmet, 2 safety-vests, 71.1ms
Speed: 31.0ms preprocess, 71.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 1 helmet, 2 safety-vests, 28.4ms
Speed: 1.6ms preprocess, 28.4ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 2 safety-vests, 31.2ms
Speed: 1.8ms preprocess, 31.2ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 31.6ms
Speed: 1.9ms preprocess, 31.6ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 30.4ms
Speed: 1.6ms preprocess, 30.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 30.3ms
Speed: 1.5ms preprocess, 30.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 2 helmets, 3 safety-vests, 24.4ms
Speed: 1.7ms preproc

In [2]:
# works well for without ppe
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
import time

# ================= CONFIG =================
MODEL_PATH = "best.pt"
VIDEO_PATH = "rd1.mp4"

CONF_THRES = 0.25
IOU_THRES = 0.12
CENTER_THRES = 0.6

PPE_BUFFER = 12
PPE_REQUIRED = 6
ALERT_COOLDOWN = 8

# ================= DANGER ZONE =================
DANGER_ZONE = np.array(
    [(276,229),(426,229),(428,210),(571,205),
     (640,430),(420,447),(417,474),(184,468)],
    dtype=np.int32
)

# ================= LOAD =================
model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

# ================= MEMORY =================
ppe_state = defaultdict(lambda: {
    "helmet": deque(maxlen=PPE_BUFFER),
    "vest": deque(maxlen=PPE_BUFFER),
    "last_seen": 0
})
last_alert = {}

# ================= HELPERS =================
def point_inside_polygon(pt, poly):
    return cv2.pointPolygonTest(poly, (int(pt[0]), int(pt[1])), False) >= 0

def iou(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[2], b[2])
    yB = min(a[3], b[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    return inter / (areaA + areaB - inter + 1e-6)

def center_inside(box, region):
    cx = (box[0] + box[2]) / 2
    cy = (box[1] + box[3]) / 2
    return region[0] <= cx <= region[2] and region[1] <= cy <= region[3]

def helmet_region(p):
    x1,y1,x2,y2 = p
    h = y2-y1
    return (x1, y1, x2, y1 + int(0.45*h))

def vest_region(p):
    x1,y1,x2,y2 = p
    h = y2-y1
    return (x1, y1 + int(0.25*h), x2, y1 + int(0.8*h))

# ================= MAIN LOOP =================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, conf=CONF_THRES)

    persons, helmets, vests = [], [], []

    for r in results:
        if r.boxes.id is None:
            continue
        for box, cls, tid in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.id):
            label = model.names[int(cls)]
            box = box.cpu().numpy().astype(int)
            tid = int(tid)

            if label == "person":
                persons.append((box, tid))
            elif label == "helmet":
                helmets.append(box)
            elif label == "safety-vest":
                vests.append(box)

    cv2.polylines(frame, [DANGER_ZONE], True, (0,0,255), 3)
    now = time.time()

    for person_box, pid in persons:
        x1,y1,x2,y2 = person_box
        foot = (int((x1+x2)//2), int(y2))

        if not point_inside_polygon(foot, DANGER_ZONE):
            continue

        hreg = helmet_region(person_box)
        vreg = vest_region(person_box)

        helmet_seen = any(
            iou(hreg,h) > IOU_THRES or center_inside(h, hreg)
            for h in helmets
        )
        vest_seen = any(
            iou(vreg,v) > IOU_THRES or center_inside(v, vreg)
            for v in vests
        )

        # -1 = occluded, 0 = missing, 1 = present
        ppe_state[pid]["helmet"].append(1 if helmet_seen else 0)
        ppe_state[pid]["vest"].append(1 if vest_seen else 0)
        ppe_state[pid]["last_seen"] = now

        helmet_ok = sum(ppe_state[pid]["helmet"]) >= PPE_REQUIRED
        vest_ok = sum(ppe_state[pid]["vest"]) >= PPE_REQUIRED

        missing = []
        if not helmet_ok:
            missing.append("Helmet")
        if not vest_ok:
            missing.append("Vest")

        violation = len(missing) > 0
        color = (0,0,255) if violation else (0,255,0)

        if violation:
            if now - last_alert.get(pid,0) > ALERT_COOLDOWN:
                print(f"🚨 ALERT ID {pid} missing {missing}")
                last_alert[pid] = now

        label = f"ID {pid} {'SAFE' if not violation else 'MISSING ' + ' & '.join(missing)}"

        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        cv2.circle(frame, foot, 5, (255,0,0), -1)
        cv2.putText(frame,label,(x1,y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)

    cv2.imshow("PPE Monitoring", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 5 persons, 1 helmet, 2 safety-vests, 65.4ms
Speed: 29.3ms preprocess, 65.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
🚨 ALERT ID 1 missing ['Helmet', 'Vest']
🚨 ALERT ID 6 missing ['Helmet', 'Vest']

0: 384x640 5 persons, 1 helmet, 2 safety-vests, 63.8ms
Speed: 1.8ms preprocess, 63.8ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 2 safety-vests, 64.7ms
Speed: 1.7ms preprocess, 64.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 66.1ms
Speed: 1.9ms preprocess, 66.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 65.5ms
Speed: 1.8ms preprocess, 65.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 68.4ms
Speed: 1.9ms preprocess, 68.4ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 64

In [ ]:
# kind of a middle ground
import cv2
import numpy as np
import time
from ultralytics import YOLO
from collections import defaultdict, deque

# ================= CONFIG =================
MODEL_PATH = "best.pt"
VIDEO_PATH = "rd1.mp4"

CONF_THRES = 0.25
IOU_THRES = 0.12

PPE_BUFFER = 20
PPE_CONFIRM = 4
MISSING_PERSIST = 15
ALERT_COOLDOWN = 10

# ================= DANGER ZONE =================
DANGER_ZONE = np.array(
    [(276,229),(426,229),(428,210),(571,205),
     (640,430),(420,447),(417,474),(184,468)],
    dtype=np.int32
)

# ================= LOAD =================
model = YOLO(MODEL_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)

# ================= MEMORY =================
ppe_memory = defaultdict(lambda: {
    "helmet": deque(maxlen=PPE_BUFFER),
    "vest": deque(maxlen=PPE_BUFFER)
})
missing_counter = defaultdict(int)
last_alert = {}

# ================= HELPERS =================
def inside_poly(pt):
    return cv2.pointPolygonTest(DANGER_ZONE, (int(pt[0]),int(pt[1])), False) >= 0

def iou(a,b):
    xA,yA = max(a[0],b[0]), max(a[1],b[1])
    xB,yB = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,xB-xA)*max(0,yB-yA)
    areaA = (a[2]-a[0])*(a[3]-a[1])
    areaB = (b[2]-b[0])*(b[3]-b[1])
    return inter/(areaA+areaB-inter+1e-6)

def center(b): return ((b[0]+b[2])//2, (b[1]+b[3])//2)

def in_box(pt,b):
    return b[0]<=pt[0]<=b[2] and b[1]<=pt[1]<=b[3]

def head_box(p):
    x1,y1,x2,y2=p; h=y2-y1; w=x2-x1
    return (int(x1-0.1*w), y1, int(x2+0.1*w), int(y1+0.4*h))

def torso_box(p):
    x1,y1,x2,y2=p; h=y2-y1; w=x2-x1
    return (int(x1-0.15*w), int(y1+0.25*h),
            int(x2+0.15*w), int(y1+0.8*h))

# ================= MAIN =================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    now = time.time()

    results = model.track(frame, persist=True, conf=CONF_THRES)

    persons, helmets, vests = [], [], []
    for r in results:
        if r.boxes.id is None: continue
        for box, cls, tid in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.id):
            box = box.cpu().numpy().astype(int)
            label = model.names[int(cls)]
            if label=="person": persons.append((box,int(tid)))
            elif label=="helmet": helmets.append(box)
            elif label=="safety-vest": vests.append(box)

    cv2.polylines(frame,[DANGER_ZONE],True,(0,0,255),3)

    for pbox,pid in persons:
        x1,y1,x2,y2=pbox
        foot=(int((x1+x2)//2),int(y2))
        if not inside_poly(foot): continue

        hbox, tbox = head_box(pbox), torso_box(pbox)

        # HELMET decision
        helmet_present = any(
            in_box(center(h), hbox) or iou(h,hbox)>IOU_THRES
            for h in helmets
        )
        helmet_absent = len(helmets)==0
        helmet_state = 1 if helmet_present else (0 if helmet_absent else -1)

        # VEST decision
        vest_present = any(
            in_box(center(v), tbox) or iou(v,tbox)>IOU_THRES
            for v in vests
        )
        vest_absent = len(vests)==0
        vest_state = 1 if vest_present else (0 if vest_absent else -1)

        ppe_memory[pid]["helmet"].append(helmet_state)
        ppe_memory[pid]["vest"].append(vest_state)

        helmet_ok = sum(1 for v in ppe_memory[pid]["helmet"] if v==1) >= PPE_CONFIRM
        vest_ok = sum(1 for v in ppe_memory[pid]["vest"] if v==1) >= PPE_CONFIRM

        missing=[]
        if not helmet_ok: missing.append("Helmet")
        if not vest_ok: missing.append("Vest")

        if missing:
            missing_counter[pid]+=1
        else:
            missing_counter[pid]=0

        violation = missing_counter[pid] > MISSING_PERSIST

        color=(0,0,255) if violation else (0,255,0)
        label=f"ID {pid} {'SAFE' if not violation else 'MISSING '+' & '.join(missing)}"

        if violation and now-last_alert.get(pid,0)>ALERT_COOLDOWN:
            print(f"🚨 ALERT ID {pid} missing {missing}")
            last_alert[pid]=now

        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        cv2.putText(frame,label,(x1,y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)

    cv2.imshow("PPE Monitoring", frame)
    if cv2.waitKey(1)==27: break

cap.release()
cv2.destroyAllWindows()



0: 384x640 5 persons, 1 helmet, 2 safety-vests, 74.1ms
Speed: 29.8ms preprocess, 74.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 1 helmet, 2 safety-vests, 65.0ms
Speed: 2.1ms preprocess, 65.0ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 2 safety-vests, 64.1ms
Speed: 2.0ms preprocess, 64.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 64.5ms
Speed: 2.2ms preprocess, 64.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 64.5ms
Speed: 2.1ms preprocess, 64.5ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 63.1ms
Speed: 1.9ms preprocess, 63.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 2 helmets, 3 safety-vests, 68.0ms
Speed: 2.1ms preproc

In [ ]:
# motion model
import cv2
import numpy as np
import time
from collections import defaultdict, deque
from ultralytics import YOLO

# =========================
# CONFIG
# =========================
YOLO_MODEL_PATH = "best.pt"          # Your YOLOv8 PPE detection model
VIDEO_PATH = "rd1.mp4"

CONF_THRES = 0.25
PPE_BUFFER = 20
PPE_CONFIRM_FRAMES = 4
MISSING_PERSIST_FRAMES = 15
ALERT_COOLDOWN_SEC = 10

# =========================
# DANGER ZONE (POLYGON)
# =========================
DANGER_ZONE = np.array(
    [(276, 229), (426, 229), (428, 210), (571, 205),
     (640, 430), (420, 447), (417, 474), (184, 468)],
    dtype=np.int32
)

# =========================
# LOAD MODELS
# =========================
model = YOLO(YOLO_MODEL_PATH)
pose_model = YOLO("yolov8n-pose.pt")  # pretrained pose model auto-download

# =========================
# VIDEO CAPTURE
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)

# =========================
# TEMPORAL MEMORY
# =========================
ppe_memory = defaultdict(lambda: {"helmet": deque(maxlen=PPE_BUFFER),
                                  "vest": deque(maxlen=PPE_BUFFER)})
missing_counter = defaultdict(int)
last_alert_time = {}

# =========================
# HELPER FUNCTIONS
# =========================
def point_inside_polygon(point, polygon):
    return cv2.pointPolygonTest(polygon, point, False) >= 0

def box_center(box):
    return (int((box[0] + box[2]) / 2), int((box[1] + box[3]) / 2))

def point_inside_box(point, box):
    x, y = point
    return box[0] <= x <= box[2] and box[1] <= y <= box[3]

def get_dynamic_ppe_zones(person_box, frame):
    x1, y1, x2, y2 = person_box
    person_crop = frame[y1:y2, x1:x2]

    results = pose_model.predict(person_crop, conf=0.3, verbose=False)
    
    # Safe check: no pose detected
    if not results or len(results[0].keypoints.xy) == 0:
        return None, None

    keypoints = results[0].keypoints.xy[0].cpu().numpy()
    if keypoints.shape[0] == 0:
        return None, None

    # Convert keypoints to full-frame coordinates
    keypoints[:, 0] += x1
    keypoints[:, 1] += y1

    # Head box (around nose / top-head)
    head = keypoints[0]  # usually nose
    head_box = (int(head[0] - 20), int(head[1] - 20),
                int(head[0] + 20), int(head[1] + 20))

    # Torso polygon (shoulders + hips)
    shoulder_left = keypoints[5]
    shoulder_right = keypoints[6]
    hip_left = keypoints[11]
    hip_right = keypoints[12]
    torso_box = np.array([shoulder_left, shoulder_right, hip_right, hip_left], dtype=np.int32)

    return head_box, torso_box

# =========================
# MAIN LOOP
# =========================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    now = time.time()

    results = model.track(frame, persist=True, conf=CONF_THRES)

    persons, helmets, vests = [], [], []

    # Organize detections
    for r in results:
        if r.boxes.id is None:
            continue

        for box, cls, tid in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.id):
            label = model.names[int(cls)]
            box = box.cpu().numpy().astype(int)
            tid = int(tid)

            if label == "person":
                persons.append((box, tid))
            elif label == "helmet":
                helmets.append(box)
            elif label == "safety-vest":
                vests.append(box)

    # Draw danger zone
    cv2.polylines(frame, [DANGER_ZONE], True, (0, 0, 255), 3)

    # Process each person
    for person_box, pid in persons:
        x1, y1, x2, y2 = person_box
        foot_point = (int((x1 + x2) / 2), int(y2))

        inside_zone = point_inside_polygon(foot_point, DANGER_ZONE)

        # Dynamic PPE zones
        head_box, torso_box = get_dynamic_ppe_zones(person_box, frame)
        if head_box is None or torso_box is None:
            continue

        # Helmet: center-point inside head_box
        has_helmet = any(point_inside_box(box_center(h), head_box) for h in helmets)

        # Vest: overlap with torso polygon
        torso_poly = torso_box.astype(np.int32)
        has_vest = False
        for v in vests:
            if cv2.pointPolygonTest(torso_poly, box_center(v), False) >= 0:
                has_vest = True
                break

        # Temporal memory
        ppe_memory[pid]["helmet"].append(has_helmet)
        ppe_memory[pid]["vest"].append(has_vest)

        helmet_ok = sum(ppe_memory[pid]["helmet"]) >= PPE_CONFIRM_FRAMES
        vest_ok = sum(ppe_memory[pid]["vest"]) >= PPE_CONFIRM_FRAMES

        missing = []
        if not helmet_ok:
            missing.append("Helmet")
        if not vest_ok:
            missing.append("Vest")

        if inside_zone and missing:
            missing_counter[pid] += 1
        else:
            missing_counter[pid] = 0

        violation = missing_counter[pid] > MISSING_PERSIST_FRAMES

        # Visualization
        if violation:
            color = (0, 0, 255)
            label = f"ID {pid}: MISSING {' & '.join(missing)}"
            last_time = last_alert_time.get(pid, 0)
            if now - last_time > ALERT_COOLDOWN_SEC:
                print(f"🚨 ALERT | ID {pid} | Missing: {missing}")
                last_alert_time[pid] = now
        else:
            color = (0, 255, 0)
            label = f"ID {pid}: SAFE"

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.circle(frame, foot_point, 5, (255, 0, 0), -1)
        cv2.putText(frame, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

        # Draw dynamic PPE zones
        cv2.rectangle(frame, head_box[:2], head_box[2:], (255, 255, 0), 1)
        cv2.polylines(frame, [torso_poly], True, (0, 255, 255), 1)

    cv2.imshow("PPE Danger Zone Monitoring", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC to quit
        break

cap.release()
cv2.destroyAllWindows()



0: 384x640 5 persons, 1 helmet, 2 safety-vests, 75.5ms
Speed: 29.9ms preprocess, 75.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 1 helmet, 2 safety-vests, 57.9ms
Speed: 2.0ms preprocess, 57.9ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 2 safety-vests, 53.2ms
Speed: 1.7ms preprocess, 53.2ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 54.0ms
Speed: 1.7ms preprocess, 54.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 57.9ms
Speed: 1.8ms preprocess, 57.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 2 helmets, 3 safety-vests, 68.0ms
Speed: 1.8ms preprocess, 68.0ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 2 helmets, 3 safety-vests, 61.5ms
Speed: 1.8ms preproc